In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm #need to install statsmodels
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import dataframe_image as dfi

In [39]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [40]:
df_assess = pd.read_csv("data/cleaned/final_assessment.csv")
df_grades = pd.read_csv("data/cleaned/grades.csv")
df_status = pd.read_csv("data/cleaned/status.csv")
df_intake = pd.read_csv("data/cleaned/intake_form.csv")

In [41]:
df_intake

,id,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,DSC 180,major,neither,Third year transfer,MATH 183,2,2,4
9,24,DSC 180,major,neither,Fourth year (second year transfer),"MATH 180A, MATH 181A, MATH 183",5,5,5


## Merge two dataframes(grades and status) by id, create coding and handwritten binary columns for b1 vs b2 test. I got rid of section 4 for now since they will affect both b1(coefficient of coding) and b2(coefficient of handwritten).

In [42]:
df_status = df_status[df_status['completed'] == 1]

sample = df_grades.merge(df_status[["id", "section"]], on="id", how="inner")

sample['coding'] =  ((sample['section'] == 2) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
sample['handwritten'] =  ((sample['section'] == 3) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
#sample_filter = sample[(sample['coding'] == 0) | (sample['handwritten'] == 0)]
sample

,id,coding_score,handwritten_score,final_score,final_score_adj,section,coding,handwritten
0,0,NaN,NaN,0.737705,0.586066,1,0,0
1,1,0.928571,NaN,0.754098,0.610656,2,1,0
2,3,0.857143,1.0,0.836066,0.479508,4,1,1
3,6,NaN,1.0,0.508197,0.200820,3,0,1
4,9,0.904764,NaN,0.672131,0.471311,2,1,0
5,11,0.928571,1.0,0.918033,0.319672,4,1,1
6,12,NaN,NaN,0.098361,0.000000,1,0,0
7,15,1.000000,1.0,0.344262,0.159836,4,1,1
8,20,NaN,NaN,0.573770,0.303279,1,0,0
9,24,NaN,NaN,1.000000,0.754098,1,0,0


## Linear model for final_score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [43]:
res = smf.ols(
    formula='final_score ~ coding * handwritten',
    data=sample
).fit()

print(res.summary())

s1 = res.summary2()
df_model = s1.tables[0]
df_coef  = s1.tables[1]
dfi.export(df_coef,  "data/statistic_analysis_outputs/final_separate.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                 -0.056
Method:                 Least Squares   F-statistic:                    0.4666
Date:                Wed, 25 Feb 2026   Prob (F-statistic):              0.708
Time:                        16:39:03   Log-Likelihood:                 3.5034
No. Observations:                  31   AIC:                            0.9932
Df Residuals:                      27   BIC:                             6.729
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.5961      0

## Linear model is final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [44]:
res2 = smf.ols(
    formula='final_score ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res2.summary())
#anova_results = anova_lm(model2, model1)
s2 = res2.summary2()
df_model2 = s2.tables[0]
df_coef2  = s2.tables[1]
dfi.export(df_coef2,  "data/statistic_analysis_outputs/final_combine.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                 -0.019
Method:                 Least Squares   F-statistic:                    0.7162
Date:                Wed, 25 Feb 2026   Prob (F-statistic):              0.497
Time:                        16:39:04   Log-Likelihood:                 3.4933
No. Observations:                  31   AIC:                           -0.9867
Df Residuals:                      28   BIC:                             3.315
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models

In [45]:
anova_results = sm.stats.anova_lm(res2, res)
print(anova_results)
dfi.export(
    anova_results,
    "data/statistic_analysis_outputs/final_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff   ss_diff         F    Pr(>F)
0      28.0  1.448797      0.0       NaN       NaN       NaN
1      27.0  1.447856      1.0  0.000941  0.017541  0.895617


## Linear model for adjusted final score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [46]:
res3 = smf.ols(
    formula='final_score_adj ~ coding * handwritten',
    data=sample
).fit()

print(res3.summary())

s3 = res3.summary2()
df_model3 = s3.tables[0]
df_coef3  = s3.tables[1]
dfi.export(df_coef3,  "data/statistic_analysis_outputs/final_adj_separate.png", table_conversion="chrome", dpi = 300)


                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.100
Method:                 Least Squares   F-statistic:                   0.09218
Date:                Wed, 25 Feb 2026   Prob (F-statistic):              0.964
Time:                        16:39:07   Log-Likelihood:                 5.3014
No. Observations:                  31   AIC:                            -2.603
Df Residuals:                      27   BIC:                             3.133
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3934      0

## Linear model is adjusted final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [47]:
res4 = smf.ols(
    formula='final_score_adj ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res4.summary())
#anova_results = anova_lm(model2, model1)
s4 = res4.summary2()
df_model4 = s4.tables[0]
df_coef4  = s4.tables[1]
dfi.export(df_coef4,  "data/statistic_analysis_outputs/final_adj_combine.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.061
Method:                 Least Squares   F-statistic:                    0.1376
Date:                Wed, 25 Feb 2026   Prob (F-statistic):              0.872
Time:                        16:39:08   Log-Likelihood:                 5.2950
No. Observations:                  31   AIC:                            -4.590
Df Residuals:                      28   BIC:                           -0.2881
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models for adjusted final score

In [48]:
anova_results2 = sm.stats.anova_lm(res4, res3)
print(anova_results2)
dfi.export(
    anova_results2,
    "data/statistic_analysis_outputs/final_adj_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff   ss_diff        F    Pr(>F)
0      28.0  1.289812      0.0       NaN      NaN       NaN
1      27.0  1.289283      1.0  0.000529  0.01108  0.916946


## Including variables in intake form

In [49]:
new_sample = sample.merge(df_intake, on="id", how="inner")
sample_new = new_sample.drop(columns = ['coding_score', 'handwritten_score', 'section'])
sample_new

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.586066,0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,0.754098,0.610656,1,0,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,0.836066,0.479508,1,1,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,0.508197,0.200820,0,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,0.672131,0.471311,1,0,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,0.918033,0.319672,1,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,0.098361,0.000000,0,0,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,0.344262,0.159836,1,1,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,0.573770,0.303279,0,0,DSC 180,major,neither,Third year transfer,MATH 183,2,2,4
9,24,1.000000,0.754098,0,0,DSC 180,major,neither,Fourth year (second year transfer),"MATH 180A, MATH 181A, MATH 183",5,5,5


In [50]:
sample_new['recruitment_source'].value_counts()
#If I remember it right, class_standing(student's year) is redundant with this feature
#since older students are likely to recruited from a uppertive class

heard_type = {
    'DSC 10': 'Heard_1',
    'DSC 20': 'Heard_1',
    'DSC 30': 'Heard_1',
    'DSC 40A': 'Heard_2',
    'DSC 40B': 'Heard_2',
    'DSC 80': 'Heard_2',
    'DSC 140B': 'Heard_3',
    'CSE 150A': 'Heard_3',
    'CSE 151A': 'Heard_3',
    'DSC 180': 'Heard_3',
    'DSC 100': 'Heard_4',
    'DSC 120': 'Heard_4',
    'DSC 190': 'Heard_4',
}

In [51]:
sample_new['dsc_affiliation'].value_counts()
dsc_type = {
    'major': 'DSC_major',
    'minor': 'DSC_minor',
    'neither': 'DSC_neither',
}

In [52]:
sample_new['math_affiliation'].value_counts()

math_type = {
    'major': 'MATH_major',
    'minor': 'MATH_minor',
    'neither': 'MATH_neither',
}

In [53]:
sample_new['class_standing'].value_counts()

class_standing
First year                            11
Third year                             7
Second year                            4
Fourth year                            4
Fourth year (second year transfer)     2
Third year (first year transfer)       1
Third year transfer                    1
third year transfer                    1
Name: count, dtype: int64

In [54]:
sample_new['stats_courses_taken'].value_counts()

section_type = {
    'MATH 181A': 'Stats_1',
    'MATH 180A': 'Stats_2',
    'ECE 109': 'Stats_2',
    'MAE 108': 'Stats_2',
    'MATH 183': 'Stats_3',
    'MATH 186': 'Stats_3',
    'ECON 120A': 'Stats_3',
    'None of the above': 'Stats_4'
}

In [55]:
def mappings(col, dictionary):
    values = [i.strip() for i in col.split(',')]
    adjust = {dictionary.get(j) for j in values if j in dictionary} #keep unique values
    return ', '.join(adjust)

In [56]:
samples = sample_new.copy()
samples['recruitment_source'] = samples['recruitment_source'].apply(lambda x: mappings(x, heard_type))
samples['stats_courses_taken'] = samples['stats_courses_taken'].apply(lambda x: mappings(x, section_type))
samples['dsc_affiliation'] = samples['dsc_affiliation'].apply(lambda x: mappings(x, dsc_type))
samples['math_affiliation'] = samples['math_affiliation'].apply(lambda x: mappings(x, math_type))
samples

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.586066,0,0,Heard_3,DSC_major,MATH_neither,Second year,"Stats_2, Stats_1",4,2,5
1,1,0.754098,0.610656,1,0,Heard_3,DSC_major,MATH_neither,Third year,Stats_2,4,3,5
2,3,0.836066,0.479508,1,1,Heard_3,DSC_major,MATH_neither,Third year,Stats_3,4,3,4
3,6,0.508197,0.200820,0,1,Heard_3,DSC_major,MATH_neither,Second year,Stats_3,3,1,4
4,9,0.672131,0.471311,1,0,Heard_3,DSC_major,MATH_major,Third year,Stats_3,4,2,4
5,11,0.918033,0.319672,1,1,Heard_3,DSC_major,MATH_neither,Second year,Stats_3,3,1,4
6,12,0.098361,0.000000,0,0,Heard_3,DSC_major,MATH_neither,Third year,Stats_3,4,4,4
7,15,0.344262,0.159836,1,1,Heard_2,DSC_major,MATH_neither,Third year (first year transfer),Stats_3,3,1,4
8,20,0.573770,0.303279,0,0,Heard_3,DSC_major,MATH_neither,Third year transfer,Stats_3,2,2,4
9,24,1.000000,0.754098,0,0,Heard_3,DSC_major,MATH_neither,Fourth year (second year transfer),"Stats_2, Stats_3, Stats_1",5,5,5


In [57]:
df_encoded = pd.concat([samples, samples['stats_courses_taken'].str.get_dummies(sep=', '), 
                        samples['recruitment_source'].str.get_dummies(sep=', '),
                        samples['dsc_affiliation'].str.get_dummies(sep=', '),
                        samples['math_affiliation'].str.get_dummies(sep=', '),], axis = 1)
df_encoded = df_encoded.drop(columns = ['recruitment_source', 'dsc_affiliation', 'math_affiliation', 'stats_courses_taken', 'Stats_4',
                           'DSC_neither', 'MATH_neither'], errors='ignore') 
#dropping all string columns and neither columns to prevent redundency
df_encoded

,id,final_score,final_score_adj,coding,handwritten,class_standing,stats_confidence,chebyshev_familiarity,python_skill_level,Stats_1,Stats_2,Stats_3,Heard_1,Heard_2,Heard_3,Heard_4,DSC_major,DSC_minor,MATH_major
0,0,0.737705,0.586066,0,0,Second year,4,2,5,1,1,0,0,0,1,0,1,0,0
1,1,0.754098,0.610656,1,0,Third year,4,3,5,0,1,0,0,0,1,0,1,0,0
2,3,0.836066,0.479508,1,1,Third year,4,3,4,0,0,1,0,0,1,0,1,0,0
3,6,0.508197,0.200820,0,1,Second year,3,1,4,0,0,1,0,0,1,0,1,0,0
4,9,0.672131,0.471311,1,0,Third year,4,2,4,0,0,1,0,0,1,0,1,0,1
5,11,0.918033,0.319672,1,1,Second year,3,1,4,0,0,1,0,0,1,0,1,0,0
6,12,0.098361,0.000000,0,0,Third year,4,4,4,0,0,1,0,0,1,0,1,0,0
7,15,0.344262,0.159836,1,1,Third year (first year transfer),3,1,4,0,0,1,0,1,0,0,1,0,0
8,20,0.573770,0.303279,0,0,Third year transfer,2,2,4,0,0,1,0,0,1,0,1,0,0
9,24,1.000000,0.754098,0,0,Fourth year (second year transfer),5,5,5,1,1,1,0,0,1,0,1,0,0


In [58]:
df_encoded.columns

Index(['id', 'final_score', 'final_score_adj', 'coding', 'handwritten', 'class_standing', 'stats_confidence', 'chebyshev_familiarity', 'python_skill_level',
       'Stats_1', 'Stats_2', 'Stats_3', 'Heard_1', 'Heard_2', 'Heard_3', 'Heard_4', 'DSC_major', 'DSC_minor', 'MATH_major'],
      dtype='object')

In [59]:
res5 = smf.ols( #no heard 1 stat, no MATH_minor in the sample
    formula='final_score ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_1 + Stats_2 + Stats_3 + Heard_2 + Heard_3 + Heard_4 + DSC_major + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res5.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.367
Model:                            OLS   Adj. R-squared:                 -0.267
Method:                 Least Squares   F-statistic:                    0.5789
Date:                Wed, 25 Feb 2026   Prob (F-statistic):              0.850
Time:                        16:39:11   Log-Likelihood:                 9.7990
No. Observations:                  31   AIC:                             12.40
Df Residuals:                      15   BIC:                             35.35
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.80

In [60]:
coeffs = pd.DataFrame({
    'coef': res5.params,
    'std_err': res5.bse,
    't': res5.tvalues,
    'p': res5.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.809752  0.276668  2.926805  0.010413
coding                 0.047698  0.158059  0.301771  0.766970
handwritten            0.048913  0.157565  0.310433  0.760504
coding:handwritten     0.070887  0.249418  0.284212  0.780134
stats_confidence       0.023029  0.064055  0.359512  0.724223
chebyshev_familiarity -0.010201  0.058454 -0.174511  0.863798
python_skill_level    -0.033686  0.078256 -0.430463  0.672980
Stats_1                0.098043  0.261891  0.374365  0.713372
Stats_2                0.016644  0.267618  0.062192  0.951231
Stats_3               -0.046602  0.168940 -0.275850  0.786427
Heard_2               -0.135067  0.258652 -0.522194  0.609161
Heard_3                0.114653  0.232007  0.494180  0.628337
Heard_4                0.321236  0.315358  1.018641  0.324523
DSC_major             -0.213783  0.222680 -0.960047  0.352252
DSC_minor             -0.136105  0.230888 -0.589482  0.564312
MATH_maj

## Adjusted Final Score

In [61]:
res6 = smf.ols( #no heard 1 stat, no MATH_minor in the sample
    formula='final_score_adj ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_1 + Stats_2 + Stats_3 + Heard_2 + Heard_3 + Heard_4 + DSC_major + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res6.summary())

#A negative adjusted R² is a warning. You probably need to:
#Reduce predictors (feature selection)
#Collect more data
#Try a different model

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.507
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     1.030
Date:                Wed, 25 Feb 2026   Prob (F-statistic):              0.478
Time:                        16:39:11   Log-Likelihood:                 16.116
No. Observations:                  31   AIC:                           -0.2325
Df Residuals:                      15   BIC:                             22.71
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.49

In [62]:
coeffs = pd.DataFrame({
    'coef': res6.params,
    'std_err': res6.bse,
    't': res6.tvalues,
    'p': res6.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.491774  0.225661  2.179264  0.045665
coding                 0.020180  0.128919  0.156534  0.877699
handwritten           -0.007631  0.128516 -0.059376  0.953437
coding:handwritten     0.001757  0.203435  0.008638  0.993222
stats_confidence       0.047990  0.052246  0.918546  0.372868
chebyshev_familiarity  0.005515  0.047677  0.115670  0.909448
python_skill_level    -0.070024  0.063828 -1.097063  0.289914
Stats_1                0.036528  0.213608  0.171004  0.866506
Stats_2                0.180097  0.218280  0.825073  0.422258
Stats_3               -0.011826  0.137794 -0.085824  0.932742
Heard_2                0.020923  0.210967  0.099177  0.922311
Heard_3                0.114231  0.189234  0.603648  0.555097
Heard_4                0.253055  0.257218  0.983818  0.340809
DSC_major             -0.170984  0.181626 -0.941409  0.361411
DSC_minor             -0.174491  0.188322 -0.926559  0.368824
MATH_maj